In [ ]:
from pathlib import Path 
import json 
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()


In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "personal_atlas"
spotify_files = list(RAW_DIR.rglob("*.json"))

spotify_files

In [ ]:
for path in spotify_files:
    print(Path)

In [ ]:
history_files = [
    path for path in spotify_files
    if "Streaming" in path.name or "Audio" in path.name
]

rows = []

for path in history_files:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        rows.extend(data)

history_df = pd.DataFrame(rows)

history_df.shape

In [ ]:
history_df.columns.tolist()

In [ ]:
history_df.head()

In [ ]:
history_df["hours_played"] = history_df["ms_played"] / 1000 / 60 / 60

top_tracks = (
    history_df
    .groupby(["master_metadata_album_artist_name", "master_metadata_track_name"], dropna=False)
    .agg(
        plays=("ms_played", "count"),
        hours_played=("hours_played", "sum"),
        total_ms=("ms_played", "sum"),
    )
    .reset_index()
    .sort_values("hours_played", ascending=False)
)

top_tracks.head(30)

In [ ]:
history_df["ts"] = pd.to_datetime(history_df["ts"], errors="coerce")

history_clean = history_df[
    history_df["master_metadata_track_name"].notna()
].copy()

history_clean.shape

In [ ]:
BTS_ARTISTS = [
    "BTS",
    "RM",
    "Jin",
    "SUGA",
    "Agust D",
    "j-hope",
    "Jimin",
    "V",
    "Jung Kook",
    "Jungkook",
]

bts_history = history_clean[
    history_clean["master_metadata_album_artist_name"].isin(BTS_ARTISTS)
].copy()

bts_history.shape

In [ ]:
bts_song_league = (
    bts_history
    .groupby(
        [
            "master_metadata_album_artist_name",
            "master_metadata_track_name",
            "master_metadata_album_album_name",
        ],
        dropna=False,
    )
    .agg(
        plays=("ms_played", "count"),
        hours_played=("hours_played", "sum"),
        total_ms=("ms_played", "sum"),
        first_play=("ts", "min"),
        last_play=("ts", "max"),
    )
    .reset_index()
    .sort_values("hours_played", ascending=False)
)

bts_song_league.head(30)

In [ ]:
bts_album_league = (
    bts_history
    .groupby(
        [
            "master_metadata_album_artist_name",
            "master_metadata_album_album_name",
        ],
        dropna=False,
    )
    .agg(
        plays=("ms_played", "count"),
        hours_played=("hours_played", "sum"),
        first_play=("ts", "min"),
        last_play=("ts", "max"),
    )
    .reset_index()
    .sort_values("hours_played", ascending=False)
)

bts_album_league.head(20)


In [ ]:
BASE_DIR = PROJECT_ROOT
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

OUT_DIR = PROCESSED_DIR / "personal_atlas"
OUT_DIR.mkdir(parents=True, exist_ok=True)

history_clean.to_parquet(OUT_DIR / "spotify_history_clean.parquet", index=False)
bts_history.to_parquet(OUT_DIR / "bts_listening_history.parquet", index=False)
bts_song_league.to_parquet(OUT_DIR / "bts_song_league.parquet", index=False)
bts_album_league.to_parquet(OUT_DIR / "bts_album_league.parquet", index=False)